# 01. Preprocessing

## Explaining the COVID-19 Effect on Mental Health Search Behavior  
### A Comparative SARIMA/SARIMAX Intervention Analysis — 6 Anglophone Countries (2004–2025)

---

### Research Objective

This project investigates how COVID-19 disrupted mental-health-related Google search behavior across six English-speaking countries. Google Trends topic-level indices are used as a proxy for population-level search interest, not as direct clinical prevalence measures.

The cleaned data from this notebook are designed to support:

1. EDA and structural-break analysis in `02_EDA.ipynb`.
2. Main anxiety modeling in `03_Modeling_COVID_Period_Final.ipynb`.
3. SARIMAX intervention variables based on the same COVID-period definition used in modeling.

---

### COVID-period definition used throughout the project

| Phase | Period |
|---|---|
| Pre-COVID | 2004-01 to 2019-12 |
| COVID period | 2020-01 to 2023-05 |
| Post-COVID | 2023-06 to 2025-12 |

Monthly timestamps are recorded on the first day of each month. For example, `2020-01-01` represents the January 2020 monthly observation.

---

### Why these 6 countries?

The six selected countries are English-speaking contexts, which reduces translation and keyword-mapping problems when using Google Trends topic-level data. This improves the comparability of search behavior across countries.

However, Google Trends values are normalized indices rather than absolute search volumes, so cross-country comparisons should focus on relative trends, structural breaks, and model-based effects rather than raw level differences.


## A. Dataset Overview

| Item | Detail |
|---|---|
| Source | Google Trends topic-level search index |
| Countries | Australia, Canada, Ireland, New Zealand, United Kingdom, United States |
| Raw period | January 2004 to December 2025 |
| Frequency | Monthly |
| Raw format | 6 separate country-level CSV files |
| Main variable | `anxiety` |
| Secondary variables | `depression`, `stress`, `insomnia`, `mental_disorder` |
| Main modeling output | `data/processed/input_modeling.csv` |

The output is a country-panel dataset. It is not a fully long-format dataset until the separate file `input_modeling_long.csv` is created in Step J.


In [25]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW_DIR = Path("data")
PROCESSED_DIR = Path("data")
EDA_OUTPUT_DIR = Path("data")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EDA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "Australia": "Australia.csv",
    "Canada": "Canada.csv",
    "Ireland": "Ireland.csv",
    "New_Zealand": "New_Zealand.csv",
    "UK": "United_Kingdom.csv",
    "US": "US.csv",
}

COLUMN_MAP = {
    "Time": "date",
    "Anxiety": "anxiety",
    "Major depressive disorder": "depression",
    "Psychological Stress": "stress",
    "Insomnia": "insomnia",
    "Mental disorder": "mental_disorder",
}

MENTAL_VARS = [
    "anxiety",
    "depression",
    "stress",
    "insomnia",
    "mental_disorder",
]

PRE_COVID_END = pd.Timestamp("2019-12-01")
COVID_START = pd.Timestamp("2020-01-01")
COVID_END = pd.Timestamp("2023-05-01")
POST_COVID_START = pd.Timestamp("2023-06-01")

print("Raw data directory:", RAW_DIR)
print("Processed output directory:", PROCESSED_DIR)
print("COVID period:", COVID_START.date(), "→", COVID_END.date())
print("Post-COVID starts:", POST_COVID_START.date())


Raw data directory: data
Processed output directory: data
COVID period: 2020-01-01 → 2023-05-01
Post-COVID starts: 2023-06-01


## B. Load and Standardize Raw Country Files

This section loads the six country-level Google Trends CSV files and standardizes column names.

Expected raw columns:

```text
Time
Anxiety
Major depressive disorder
Psychological Stress
Insomnia
Mental disorder
```

Standardized columns:

```text
date, country, anxiety, depression, stress, insomnia, mental_disorder
```


In [26]:
def load_country_file(country: str, filename: str) -> pd.DataFrame:
    path = RAW_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing raw file: {path}. Put the raw CSV files inside data."
        )

    df = pd.read_csv(path)
    missing_cols = set(COLUMN_MAP.keys()) - set(df.columns)
    if missing_cols:
        raise ValueError(f"{filename} is missing columns: {sorted(missing_cols)}")

    df = df.rename(columns=COLUMN_MAP)
    df = df[list(COLUMN_MAP.values())].copy()
    df["country"] = country

    df["date"] = pd.to_datetime(df["date"])
    for col in MENTAL_VARS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    df = df[["date", "country"] + MENTAL_VARS]
    df = df.sort_values("date").reset_index(drop=True)
    return df

raw_frames = []
for country, filename in RAW_FILES.items():
    country_df = load_country_file(country, filename)
    raw_frames.append(country_df)
    print(f"{country:<12} {country_df.shape[0]:>3} rows | {country_df['date'].min().date()} → {country_df['date'].max().date()}")

raw_panel = pd.concat(raw_frames, ignore_index=True)
print("\nRaw panel shape:", raw_panel.shape)
display(raw_panel.head())


Australia    264 rows | 2004-01-01 → 2025-12-01
Canada       264 rows | 2004-01-01 → 2025-12-01
Ireland      264 rows | 2004-01-01 → 2025-12-01
New_Zealand  264 rows | 2004-01-01 → 2025-12-01
UK           264 rows | 2004-01-01 → 2025-12-01
US           264 rows | 2004-01-01 → 2025-12-01

Raw panel shape: (1584, 7)


,date,country,anxiety,depression,stress,insomnia,mental_disorder
0,2004-01-01,Australia,27.0,55.0,28.0,6.0,16.0
1,2004-02-01,Australia,31.0,63.0,34.0,9.0,28.0
2,2004-03-01,Australia,32.0,82.0,45.0,9.0,40.0
3,2004-04-01,Australia,33.0,73.0,45.0,5.0,34.0
4,2004-05-01,Australia,33.0,87.0,49.0,8.0,38.0


## C. Validate Raw Monthly Panel Structure

The raw dataset should contain:

\[
6 countries * 264 months = 1584 observations
\]

The expected raw period is January 2004 to December 2025.


In [27]:
expected_start = pd.Timestamp("2004-01-01")
expected_end = pd.Timestamp("2025-12-01")
expected_months = len(pd.date_range(expected_start, expected_end, freq="MS"))

validation_rows = []
for country, g in raw_panel.groupby("country"):
    dates = pd.DatetimeIndex(g["date"])
    expected_dates = pd.date_range(expected_start, expected_end, freq="MS")
    missing_dates = sorted(set(expected_dates) - set(dates))
    duplicate_dates = g["date"][g["date"].duplicated()].tolist()

    validation_rows.append({
        "country": country,
        "n_rows": len(g),
        "start_date": g["date"].min(),
        "end_date": g["date"].max(),
        "missing_months": len(missing_dates),
        "duplicate_months": len(duplicate_dates),
        "passes_structure_check": (
            len(g) == expected_months and
            g["date"].min() == expected_start and
            g["date"].max() == expected_end and
            len(missing_dates) == 0 and
            len(duplicate_dates) == 0
        )
    })

raw_validation = pd.DataFrame(validation_rows)
display(raw_validation)

if not raw_validation["passes_structure_check"].all():
    raise ValueError("Raw panel validation failed. Check missing/duplicate monthly observations.")


,country,n_rows,start_date,end_date,missing_months,duplicate_months,passes_structure_check
0,Australia,264,2004-01-01,2025-12-01,0,0,True
1,Canada,264,2004-01-01,2025-12-01,0,0,True
2,Ireland,264,2004-01-01,2025-12-01,0,0,True
3,New_Zealand,264,2004-01-01,2025-12-01,0,0,True
4,UK,264,2004-01-01,2025-12-01,0,0,True
5,US,264,2004-01-01,2025-12-01,0,0,True


## D. Handle Google Trends Zero Values

Google Trends zeros are treated cautiously. A zero does not necessarily mean that search interest was truly zero; it can reflect very low or insufficient search volume relative to the query scale.

Preprocessing rule used here:

1. Convert zeros in the five Google Trends variables to missing values.
2. Interpolate short missing gaps using linear interpolation.
3. Limit interpolation to at most 3 consecutive months in either direction.
4. Leave longer leading gaps unresolved in `input.csv` for transparency.

This rule avoids replacing long early low-volume periods too aggressively.


In [28]:
zero_summary_rows = []
raw_missing_or_zero_flags = raw_panel[["date", "country"]].copy()

clean_panel = raw_panel.copy()

for var in MENTAL_VARS:
    flag_col = f"__{var}_raw_missing_or_zero"
    raw_missing_or_zero_flags[flag_col] = (
        clean_panel[var].isna() | clean_panel[var].eq(0)
    ).astype(int)

for country, g in clean_panel.groupby("country"):
    for var in MENTAL_VARS:
        zero_summary_rows.append({
            "country": country,
            "variable": var,
            "zero_count_raw": int(g[var].eq(0).sum()),
            "missing_count_raw": int(g[var].isna().sum()),
        })

zero_summary = pd.DataFrame(zero_summary_rows)
zero_summary.to_csv(EDA_OUTPUT_DIR / "01_preprocessing_zero_summary.csv", index=False)

display(zero_summary.query("zero_count_raw > 0 or missing_count_raw > 0"))

# Convert zero values to NaN before interpolation.
for var in MENTAL_VARS:
    clean_panel[var] = clean_panel[var].mask(clean_panel[var].eq(0), np.nan)


,country,variable,zero_count_raw,missing_count_raw
10,Ireland,anxiety,4,0
12,Ireland,stress,3,0
13,Ireland,insomnia,26,0
14,Ireland,mental_disorder,4,0
17,New_Zealand,stress,3,0
18,New_Zealand,insomnia,22,0
22,UK,stress,2,0


## E. Short-Gap Linear Interpolation

The interpolation rule is intentionally conservative:

```python
interpolate(method="linear", limit=3, limit_direction="both")
```

Meaning:

- Short internal gaps are filled linearly.
- Short leading/trailing gaps up to 3 months are filled from the nearest observed value.
- Long leading gaps remain missing in `input.csv`.

This produces a transparent EDA dataset and avoids over-imputing long low-volume early periods.


In [29]:
interpolated_panel = clean_panel.copy()

missing_before = interpolated_panel[MENTAL_VARS].isna().sum()

for country in sorted(interpolated_panel["country"].unique()):
    m = interpolated_panel["country"].eq(country)
    for var in MENTAL_VARS:
        interpolated_panel.loc[m, var] = (
            interpolated_panel.loc[m, var]
            .interpolate(method="linear", limit=3, limit_direction="both")
        )

missing_after = interpolated_panel[MENTAL_VARS].isna().sum()

missing_summary = pd.DataFrame({
    "missing_after_zero_to_nan": missing_before,
    "missing_after_short_gap_interpolation": missing_after,
})

display(missing_summary)

remaining_missing = interpolated_panel[interpolated_panel[MENTAL_VARS].isna().any(axis=1)]
print("Remaining missing observations after short-gap interpolation:", len(remaining_missing))
display(remaining_missing[["date", "country"] + MENTAL_VARS])


,missing_after_zero_to_nan,missing_after_short_gap_interpolation
anxiety,4,0
depression,0,0
stress,8,0
insomnia,48,6
mental_disorder,4,0


Remaining missing observations after short-gap interpolation: 6


,date,country,anxiety,depression,stress,insomnia,mental_disorder
792,2004-01-01,New_Zealand,25.0,53.0,45.0,NaN,33.0
793,2004-02-01,New_Zealand,21.0,53.0,48.0,NaN,25.0
794,2004-03-01,New_Zealand,41.0,54.0,51.0,NaN,28.0
795,2004-04-01,New_Zealand,37.0,63.0,63.0,NaN,40.0
796,2004-05-01,New_Zealand,27.0,83.0,68.0,NaN,43.0
797,2004-06-01,New_Zealand,27.0,81.0,53.0,NaN,26.0


## F. Save Clean EDA Dataset: `input.csv`

`input.csv` is the clean country-panel file used for EDA transparency. It may still contain a small number of long leading missing values if they were not safely interpolated.

Expected columns:

```text
date, country, anxiety, depression, stress, insomnia, mental_disorder
```


In [30]:
input_csv = interpolated_panel[["date", "country"] + MENTAL_VARS].copy()
input_csv = input_csv.sort_values(["country", "date"]).reset_index(drop=True)

INPUT_PATH = PROCESSED_DIR / "01_input.csv"
input_csv.to_csv(INPUT_PATH, index=False)

print("Saved:", INPUT_PATH)
print("Shape:", input_csv.shape)
print("Total remaining NaN:", int(input_csv[MENTAL_VARS].isna().sum().sum()))
display(input_csv.head())


Saved: data\01_input.csv
Shape: (1584, 7)
Total remaining NaN: 6


,date,country,anxiety,depression,stress,insomnia,mental_disorder
0,2004-01-01,Australia,27.0,55.0,28.0,6.0,16.0
1,2004-02-01,Australia,31.0,63.0,34.0,9.0,28.0
2,2004-03-01,Australia,32.0,82.0,45.0,9.0,40.0
3,2004-04-01,Australia,33.0,73.0,45.0,5.0,34.0
4,2004-05-01,Australia,33.0,87.0,49.0,8.0,38.0


## G. Create Modeling-Ready Dataset: `input_modeling.csv`

The SARIMA/SARIMAX modeling notebook requires complete monthly series. Therefore, any remaining edge missing values after conservative interpolation are filled using within-country forward/backward filling.

Important distinction:

| File | Missing values allowed? | Purpose |
|---|---:|---|
| `input.csv` | Yes, if long leading low-volume gaps remain | transparent EDA dataset |
| `input_modeling.csv` | No | SARIMA/SARIMAX modeling |

The modeling file contains imputation flags so that all replaced low-volume/missing observations are traceable.


In [31]:
modeling_df = interpolated_panel.merge(
    raw_missing_or_zero_flags,
    on=["date", "country"],
    how="left"
)

# Create final imputation flags before edge filling
# A flag equals 1 if the raw observation was zero/missing or if it remained missing after interpolation
for var in MENTAL_VARS:
    temp_flag = f"__{var}_raw_missing_or_zero"
    modeling_df[f"{var}_was_imputed"] = (
        modeling_df[temp_flag].eq(1) | modeling_df[var].isna()
    ).astype(int)

# Fill any remaining edge missing values within each country
for var in MENTAL_VARS:
    modeling_df[var] = (
        modeling_df.groupby("country")[var]
        .transform(lambda s: s.ffill().bfill())
    )

# Drop temporary raw flags
modeling_df = modeling_df.drop(columns=[c for c in modeling_df.columns if c.startswith("__")])

modeling_df["year"] = modeling_df["date"].dt.year
modeling_df["month"] = modeling_df["date"].dt.month
modeling_df["time_index"] = modeling_df.groupby("country").cumcount()

# Phase labels
def assign_phase(date):
    if date <= PRE_COVID_END:
        return "pre_covid"
    if COVID_START <= date <= COVID_END:
        return "covid_period"
    return "post_covid"

modeling_df["analysis_phase"] = modeling_df["date"].apply(assign_phase)

# SARIMAX intervention variables aligned with 03_Modeling.ipynb
modeling_df["covid_period"] = (
    (modeling_df["date"] >= COVID_START) &
    (modeling_df["date"] <= COVID_END)
).astype(int)

modeling_df["post_covid_period"] = (
    modeling_df["date"] >= POST_COVID_START
).astype(int)

modeling_df["covid_trend"] = 0
modeling_df["post_covid_trend"] = 0

for country in sorted(modeling_df["country"].unique()):
    m_country = modeling_df["country"].eq(country)

    m_covid = (
        m_country &
        (modeling_df["date"] >= COVID_START) &
        (modeling_df["date"] <= COVID_END)
    )
    modeling_df.loc[m_covid, "covid_trend"] = np.arange(1, m_covid.sum() + 1)

    m_post = m_country & (modeling_df["date"] >= POST_COVID_START)
    modeling_df.loc[m_post, "post_covid_trend"] = np.arange(1, m_post.sum() + 1)

flag_cols = [f"{var}_was_imputed" for var in MENTAL_VARS]
modeling_cols = (
    ["date", "country"] +
    MENTAL_VARS +
    flag_cols +
    ["year", "month", "time_index", "analysis_phase",
     "covid_period", "covid_trend", "post_covid_period", "post_covid_trend"]
)
modeling_df = modeling_df[modeling_cols].sort_values(["country", "date"]).reset_index(drop=True)

# Validation.
if modeling_df[MENTAL_VARS].isna().sum().sum() != 0:
    raise ValueError("input_modeling.csv still contains missing values in modeling variables.")

INPUT_MODELING_PATH = PROCESSED_DIR / "01_input_modeling.csv"
modeling_df.to_csv(INPUT_MODELING_PATH, index=False)

print("Saved:", INPUT_MODELING_PATH)
print("Shape:", modeling_df.shape)
print("Total NaN:", int(modeling_df.isna().sum().sum()))
print("COVID months per country:", modeling_df.groupby("country")["covid_period"].sum().to_dict())
print("Post-COVID months per country:", modeling_df.groupby("country")["post_covid_period"].sum().to_dict())
display(modeling_df.head())


Saved: data\01_input_modeling.csv
Shape: (1584, 20)
Total NaN: 0
COVID months per country: {'Australia': 41, 'Canada': 41, 'Ireland': 41, 'New_Zealand': 41, 'UK': 41, 'US': 41}
Post-COVID months per country: {'Australia': 31, 'Canada': 31, 'Ireland': 31, 'New_Zealand': 31, 'UK': 31, 'US': 31}


,date,country,anxiety,depression,stress,insomnia,mental_disorder,anxiety_was_imputed,depression_was_imputed,stress_was_imputed,insomnia_was_imputed,mental_disorder_was_imputed,year,month,time_index,analysis_phase,covid_period,covid_trend,post_covid_period,post_covid_trend
0,2004-01-01,Australia,27.0,55.0,28.0,6.0,16.0,0,0,0,0,0,2004,1,0,pre_covid,0,0,0,0
1,2004-02-01,Australia,31.0,63.0,34.0,9.0,28.0,0,0,0,0,0,2004,2,1,pre_covid,0,0,0,0
2,2004-03-01,Australia,32.0,82.0,45.0,9.0,40.0,0,0,0,0,0,2004,3,2,pre_covid,0,0,0,0
3,2004-04-01,Australia,33.0,73.0,45.0,5.0,34.0,0,0,0,0,0,2004,4,3,pre_covid,0,0,0,0
4,2004-05-01,Australia,33.0,87.0,49.0,8.0,38.0,0,0,0,0,0,2004,5,4,pre_covid,0,0,0,0


## H. Validate Modeling Intervention Variables

This validation confirms that each country has:

- 192 pre-COVID months from 2004-01 to 2019-12.
- 41 COVID-period months from 2020-01 to 2023-05.
- 31 post-COVID months from 2023-06 to 2025-12.

These counts match the split and intervention design used in `03_Modeling_COVID_Period_Final.ipynb`.


In [32]:
phase_validation = (
    modeling_df
    .groupby(["country", "analysis_phase"])
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        n_months=("date", "size"),
        covid_period_sum=("covid_period", "sum"),
        covid_trend_max=("covid_trend", "max"),
        post_covid_period_sum=("post_covid_period", "sum"),
        post_covid_trend_max=("post_covid_trend", "max"),
    )
    .reset_index()
)

display(phase_validation)
phase_validation.to_csv(EDA_OUTPUT_DIR / "01_preprocessing_phase_validation.csv", index=False)

expected_phase_counts = {"pre_covid": 192, "covid_period": 41, "post_covid": 31}
for _, row in phase_validation.iterrows():
    expected = expected_phase_counts[row["analysis_phase"]]
    if row["n_months"] != expected:
        raise ValueError(f"Unexpected phase count for {row['country']} {row['analysis_phase']}: {row['n_months']} vs {expected}")


,country,analysis_phase,start_date,end_date,n_months,covid_period_sum,covid_trend_max,post_covid_period_sum,post_covid_trend_max
0,Australia,covid_period,2020-01-01,2023-05-01,41,41,41,0,0
1,Australia,post_covid,2023-06-01,2025-12-01,31,0,0,31,31
2,Australia,pre_covid,2004-01-01,2019-12-01,192,0,0,0,0
3,Canada,covid_period,2020-01-01,2023-05-01,41,41,41,0,0
4,Canada,post_covid,2023-06-01,2025-12-01,31,0,0,31,31
5,Canada,pre_covid,2004-01-01,2019-12-01,192,0,0,0,0
6,Ireland,covid_period,2020-01-01,2023-05-01,41,41,41,0,0
7,Ireland,post_covid,2023-06-01,2025-12-01,31,0,0,31,31
8,Ireland,pre_covid,2004-01-01,2019-12-01,192,0,0,0,0
9,New_Zealand,covid_period,2020-01-01,2023-05-01,41,41,41,0,0


## I. Create Long-Format Modeling Dataset: `input_modeling_long.csv`

`input_modeling_long.csv` is useful for EDA, plotting, heatmaps, and robustness comparisons across variables.

Wide format:

```text
date | country | anxiety | depression | stress | insomnia | mental_disorder | ...
```

Long format:

```text
date | country | variable | value | was_imputed | ...
```


In [33]:
id_vars = [
    "date", "country", "year", "month", "time_index", "analysis_phase",
    "covid_period", "covid_trend", "post_covid_period", "post_covid_trend",
]

value_long = modeling_df.melt(
    id_vars=id_vars,
    value_vars=MENTAL_VARS,
    var_name="variable",
    value_name="value",
)

flag_long = modeling_df.melt(
    id_vars=["date", "country"],
    value_vars=flag_cols,
    var_name="flag_variable",
    value_name="was_imputed",
)
flag_long["variable"] = flag_long["flag_variable"].str.replace("_was_imputed", "", regex=False)
flag_long = flag_long.drop(columns=["flag_variable"])

modeling_long = value_long.merge(
    flag_long,
    on=["date", "country", "variable"],
    how="left",
)

modeling_long = modeling_long[
    ["date", "country", "variable", "value", "was_imputed"] +
    [c for c in id_vars if c not in ["date", "country"]]
]

INPUT_MODELING_LONG_PATH = PROCESSED_DIR / "01_input_modeling_long.csv"
modeling_long.to_csv(INPUT_MODELING_LONG_PATH, index=False)

print("Saved:", INPUT_MODELING_LONG_PATH)
print("Shape:", modeling_long.shape)
print("Total NaN:", int(modeling_long.isna().sum().sum()))
display(modeling_long.head())


Saved: data\01_input_modeling_long.csv
Shape: (7920, 13)
Total NaN: 0


,date,country,variable,value,was_imputed,year,month,time_index,analysis_phase,covid_period,covid_trend,post_covid_period,post_covid_trend
0,2004-01-01,Australia,anxiety,27.0,0,2004,1,0,pre_covid,0,0,0,0
1,2004-02-01,Australia,anxiety,31.0,0,2004,2,1,pre_covid,0,0,0,0
2,2004-03-01,Australia,anxiety,32.0,0,2004,3,2,pre_covid,0,0,0,0
3,2004-04-01,Australia,anxiety,33.0,0,2004,4,3,pre_covid,0,0,0,0
4,2004-05-01,Australia,anxiety,33.0,0,2004,5,4,pre_covid,0,0,0,0


## J. Modeling Data Dictionary

This section creates a detailed markdown data dictionary. It explains the modeling-ready files, variable definitions, COVID intervention design, imputation flags, and the connection between preprocessing and modeling.


In [34]:
imputation_counts = (
    modeling_df
    .groupby("country")[[f"{v}_was_imputed" for v in MENTAL_VARS]]
    .sum()
    .astype(int)
)

imputation_table_md = imputation_counts.reset_index().to_markdown(index=False)

variable_table = pd.DataFrame([
    ["date", "Monthly timestamp stored as first day of month. Example: 2020-01-01 represents January 2020."],
    ["country", "Country identifier: Australia, Canada, Ireland, New_Zealand, UK, or US."],
    ["anxiety", "Google Trends topic-level index for Anxiety."],
    ["depression", "Google Trends topic-level index for Major depressive disorder."],
    ["stress", "Google Trends topic-level index for Psychological Stress."],
    ["insomnia", "Google Trends topic-level index for Insomnia."],
    ["mental_disorder", "Google Trends topic-level index for Mental disorder."],
    ["*_was_imputed", "Flag equal to 1 if the corresponding raw value was zero/missing and was filled during preprocessing/modeling preparation."],
    ["year", "Calendar year extracted from date."],
    ["month", "Calendar month extracted from date."],
    ["time_index", "Monthly time index within each country, starting at 0 in 2004-01."],
    ["analysis_phase", "Categorical phase: pre_covid, covid_period, or post_covid."],
    ["covid_period", "Dummy equal to 1 from 2020-01 to 2023-05, otherwise 0."],
    ["covid_trend", "Monthly counter inside COVID period: 1, 2, ..., 41; 0 outside COVID period."],
    ["post_covid_period", "Dummy equal to 1 from 2023-06 onward, otherwise 0."],
    ["post_covid_trend", "Monthly counter inside post-COVID period: 1, 2, ..., 31; 0 before post-COVID period."],
], columns=["Variable", "Definition"])

variable_table_md = variable_table.to_markdown(index=False)

dictionary_text = f"""# Modeling Data Dictionary

## Project

**Explaining the COVID-19 Effect on Mental Health Search Behavior: A Comparative SARIMA/SARIMAX Intervention Analysis Across Six Anglophone Countries.**

This dictionary describes the modeling-ready data created by `01_Preprocessing.ipynb` and used by `03_Modeling_COVID_Period_Final.ipynb`.

---

## Files created

| File | Format | Purpose |
|---|---|---|
| `input.csv` | wide country-panel | transparent cleaned EDA dataset; may retain long leading missing values |
| `input_modeling.csv` | wide country-panel | complete modeling dataset for ARIMA, SARIMA, and SARIMAX |
| `input_modeling_long.csv` | long format | plotting, cross-country/variable comparisons, robustness summaries |
| `modeling_data_dictionary.md` | markdown | documentation of variables, imputation, COVID intervention design, and model alignment |

---

## Sample coverage

| Item | Value |
|---|---|
| Countries | Australia, Canada, Ireland, New_Zealand, UK, US |
| Frequency | Monthly |
| Start | 2004-01 |
| End | 2025-12 |
| Months per country | 264 |
| Total rows in `input_modeling.csv` | 1,584 |
| Total rows in `input_modeling_long.csv` | 7,920 |

---

## COVID-period definition

| Phase | Period | Months per country | Modeling interpretation |
|---|---:|---:|---|
| Pre-COVID | 2004-01 to 2019-12 | 192 | baseline period |
| COVID period | 2020-01 to 2023-05 | 41 | disruption/intervention period |
| Post-COVID | 2023-06 to 2025-12 | 31 | post-intervention adjustment period |

This phase definition must remain identical across preprocessing, EDA, and modeling.

---

## Variable definitions

{variable_table_md}

---

## Imputation policy

Google Trends zeros are treated as potentially low-volume or insufficient-search-volume observations rather than literal absence of search interest.

Preprocessing uses this sequence:

1. Convert zeros in Google Trends variables to missing values.
2. Apply linear interpolation to short gaps using `limit=3` and `limit_direction='both'`.
3. Save the transparent EDA dataset as `input.csv`.
4. Fill any remaining edge missing values within each country using forward/backward fill for `input_modeling.csv`.
5. Create variable-specific imputation flags before final modeling fill.

Interpretation of imputation flags:

- `0`: value was directly observed as a nonzero Google Trends value.
- `1`: value was originally zero/missing and was filled during preprocessing/modeling preparation.

### Imputed observation counts by country

{imputation_table_md}

---

## SARIMAX intervention variables

The modeling notebook uses the following exogenous variables:

| Variable | Role in SARIMAX | Interpretation |
|---|---|---|
| `covid_period` | level intervention | average level shift during COVID period |
| `covid_trend` | slope intervention | month-by-month persistence or fading during COVID period |
| `post_covid_period` | level intervention | average level shift after COVID period ends |
| `post_covid_trend` | slope intervention | post-COVID recovery, normalization, or continued increase |

The trend counters use monthly units. For example, `covid_trend = 12` means the 12th month of the COVID period.

---

## Modeling alignment

`03_Modeling.ipynb` uses:

| Split | Period | Purpose |
|---|---|---|
| Train | 2004-01 to 2021-12 | estimate candidate model parameters |
| Validation | 2022-01 to 2023-05 | select ARIMA/SARIMA/SARIMAX specification |
| Test | 2023-06 to 2025-12 | final out-of-sample forecast evaluation |
| Counterfactual pre-COVID fit | 2004-01 to 2019-12 | estimate no-COVID SARIMA baseline |
| Counterfactual forecast | 2020-01 to 2025-12 | compare observed data with no-COVID forecast |

---

## Candidate-order logic from EDA

The EDA ACF/PACF analysis motivates the model candidate set used in `03_Modeling_COVID_Period_Final.ipynb`:

| EDA finding | Modeling implication |
|---|---|
| First-differenced anxiety is more stationary than levels | set non-seasonal `d = 1` |
| ACF has strong lag-1 dependence | include `ARIMA(0,1,1)` |
| ACF and PACF both show short-lag signals | include `ARIMA(1,1,1)` |
| Some countries show lag 1–2 dependence/noisier short-run structure | include `ARIMA(2,1,1)` and `ARIMA(1,1,2)` |
| ACF/PACF show seasonal spikes at 12, 24, 36 | use monthly seasonal period `s = 12` |
| Seasonal differencing is plausible but not automatic | compare seasonal `D = 0` and `D = 1` |
| Seasonal AR/MA behavior appears in ACF/PACF | test seasonal AR and MA terms |

Final model selection is not based on visual ACF/PACF alone. It is based on validation forecast errors, test forecast errors, AIC/BIC, and residual diagnostics.
"""

DICTIONARY_PATH = PROCESSED_DIR / "01_modeling_data_dictionary.md"
DICTIONARY_PATH.write_text(dictionary_text, encoding="utf-8")

print("Saved:", DICTIONARY_PATH)
print(dictionary_text[:1200])


Saved: data\01_modeling_data_dictionary.md
# Modeling Data Dictionary

## Project

**Explaining the COVID-19 Effect on Mental Health Search Behavior: A Comparative SARIMA/SARIMAX Intervention Analysis Across Six Anglophone Countries.**

This dictionary describes the modeling-ready data created by `01_Preprocessing.ipynb` and used by `03_Modeling_COVID_Period_Final.ipynb`.

---

## Files created

| File | Format | Purpose |
|---|---|---|
| `input.csv` | wide country-panel | transparent cleaned EDA dataset; may retain long leading missing values |
| `input_modeling.csv` | wide country-panel | complete modeling dataset for ARIMA, SARIMA, and SARIMAX |
| `input_modeling_long.csv` | long format | plotting, cross-country/variable comparisons, robustness summaries |
| `modeling_data_dictionary.md` | markdown | documentation of variables, imputation, COVID intervention design, and model alignment |

---

## Sample coverage

| Item | Value |
|---|---|
| Countries | Australia, Canada, Ireland, N

## K. Final Output Checklist

This cell confirms that all preprocessing outputs required by EDA and modeling have been created.


In [35]:
output_files = [
    PROCESSED_DIR / "01_input.csv",
    PROCESSED_DIR / "01_input_modeling.csv",
    PROCESSED_DIR / "01_input_modeling_long.csv",
    PROCESSED_DIR / "01_modeling_data_dictionary.md",
    EDA_OUTPUT_DIR / "01_preprocessing_zero_summary.csv",
    EDA_OUTPUT_DIR / "01_preprocessing_phase_validation.csv",
]

checklist = pd.DataFrame({
    "file": [str(p) for p in output_files],
    "exists": [p.exists() for p in output_files],
})

display(checklist)

if not checklist["exists"].all():
    raise FileNotFoundError("Some expected preprocessing outputs were not created.")

print("Preprocessing completed successfully.")


,file,exists
0,data\01_input.csv,True
1,data\01_input_modeling.csv,True
2,data\01_input_modeling_long.csv,True
3,data\01_modeling_data_dictionary.md,True
4,data\01_preprocessing_zero_summary.csv,True
5,data\01_preprocessing_phase_validation.csv,True


Preprocessing completed successfully.
